# 공통 코드

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib

from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import MinMaxScaler

pd.set_option('display.max_rows', None)      # 모든 행 출력
pd.set_option('display.max_columns', None)   # 모든 열 출력
pd.set_option('display.width', None)         # 출력 너비 제한 해제
pd.set_option('display.max_colwidth', None)  # 컬럼 내용 길이 제한 해제

In [ ]:
# 입력 데이터
DATA_DIR = Path("processed_data/psf_tables/number")

# GMM 저장 위치
GMM_DIR = Path("models/gmm")
GMM_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# 공통 컬럼 정리 함수
def clean_columns(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.replace("\u00a0", " ", regex=False)
        .str.strip()
    )
    return df

In [ ]:
def make_composite_state(
    df,
    group_cols,
    state_name,
    n_components=3,
    random_state=42,
):
    result = df.copy()

    X = result[group_cols].copy()

    X = X.apply(
        pd.to_numeric,
        errors="coerce",
    )

    # API에서도 동일하게 사용할 결측치 대체값
    fill_values = {}

    for col in group_cols:
        if X[col].isna().any():

            mode = X[col].mode(
                dropna=True
            )

            if mode.empty:
                raise ValueError(
                    f"{col}의 유효값이 없습니다."
                )

            fill_value = mode.iloc[0]

            X[col] = X[col].fillna(
                fill_value
            )

            fill_values[col] = float(
                fill_value
            )

        else:
            # 결측이 없더라도 API용으로 최빈값 저장
            mode = X[col].mode(
                dropna=True
            )

            if not mode.empty:
                fill_values[col] = float(
                    mode.iloc[0]
                )

    # ----------------------------------------
    # 단일 변수
    # ----------------------------------------

    if len(group_cols) == 1:

        col = group_cols[0]

        result[state_name] = X[col]

        return {
            "result": result,
            "gmm": None,
            "label_map": None,
            "group_cols": group_cols,
            "fill_values": fill_values,
        }

    # ----------------------------------------
    # GMM
    # ----------------------------------------

    gmm = GaussianMixture(
        n_components=n_components,
        covariance_type="full",
        random_state=random_state,
        n_init=20,
    )

    cluster = gmm.fit_predict(X)

    cluster_mean = pd.DataFrame(
        gmm.means_,
        columns=group_cols,
    )

    cluster_mean["mean_score"] = (
        cluster_mean[group_cols]
        .mean(axis=1)
    )

    ordered_clusters = (
        cluster_mean
        .sort_values("mean_score")
        .index
        .tolist()
    )

    label_map = {
        int(cluster_id): int(label)
        for label, cluster_id
        in enumerate(ordered_clusters)
    }

    result[state_name] = [
        label_map[int(c)]
        for c in cluster
    ]

    result[
        state_name + "_cluster"
    ] = cluster

    return {
        "result": result,
        "gmm": gmm,
        "label_map": label_map,
        "group_cols": list(group_cols),
        "fill_values": fill_values,
        "cluster_mean": cluster_mean,
    }

In [ ]:
GMM_GROUPS = {

    "PSF2": {
        "Equipment_Quality": [
            "작업 도구의 적절성",
            "장비의 사용성",
        ],

        "System_Support": [
            "컴퓨터 지원 시스템의 적절성",
            "타 기기와의 연계성",
        ],
    },

    "PSF4": {
        "Training_Frequency": [
            "교육 주기(빈도)",
            "직무능력 평가 빈도",
        ],

        "Technical_Knowledge": [
            "시스템 지식(원리적 지식)의 숙련도",
            "장비 사용 지식의 숙련도",
        ],
    },

    "PSF6": {
        "Thermal_Comfort": [
            "실내 온도",
            "실내 습도",
            "통풍",
        ],

        "Visual_Environment": [
            "업무공간 가시성",
            "안개",
            "황사",
        ],

        "Surface_Safety": [
            "마찰계수",
            "비",
        ],

        "Outdoor_Climate": [
            "실외 온도",
            "실외 습도",
            "풍속",
            "낙뢰",
            "비",
        ],
    },

    "PSF8": {
        "Decision_Complexity": [
            "의사결정 대안의 수",
            "의사결정 판단 기준의 수",
        ],

        "Information_Complexity": [
            "관측되는 정보의 양",
            "(관측이 불가능하여)추론해야하는 정보의 양",
        ],
    },
}

In [ ]:
def build_and_save_psf_gmm(
    df,
    psf_name,
    group_config,
):

    bundle = {}

    working_df = df.copy()

    for state_name, group_cols in group_config.items():

        print(
            f"[{psf_name}] {state_name}"
        )

        output = make_composite_state(
            df=working_df,
            group_cols=group_cols,
            state_name=state_name,
        )

        working_df = output[
            "result"
        ]

        bundle[state_name] = {
            "gmm": output["gmm"],
            "label_map": output["label_map"],
            "group_cols": output["group_cols"],
            "fill_values": output["fill_values"],
        }

        if output["gmm"] is not None:

            print(
                " label_map:",
                output["label_map"],
            )

            display(
                output["cluster_mean"]
            )

    save_path = (
        GMM_DIR
        / f"{psf_name.lower()}_gmm.joblib"
    )

    joblib.dump(
        bundle,
        save_path,
    )

    print(
        f"\n저장 완료: {save_path}"
    )

    return working_df, bundle

In [ ]:
psf2_df = pd.read_csv(
    DATA_DIR / "장비_및_기기.csv"
)

psf2_result, psf2_bundle = (
    build_and_save_psf_gmm(
        psf2_df,
        "PSF2",
        GMM_GROUPS["PSF2"],
    )
)

In [ ]:
psf4_df = pd.read_csv(
    DATA_DIR / "교육_및_훈련.csv"
)

psf4_result, psf4_bundle = (
    build_and_save_psf_gmm(
        psf4_df,
        "PSF4",
        GMM_GROUPS["PSF4"],
    )
)

In [ ]:
psf6_df = pd.read_csv(
    DATA_DIR / "환경.csv"
)

psf6_result, psf6_bundle = (
    build_and_save_psf_gmm(
        psf6_df,
        "PSF6",
        GMM_GROUPS["PSF6"],
    )
)

In [ ]:
psf8_df = pd.read_csv(
    DATA_DIR / "복잡성.csv"
)

psf8_result, psf8_bundle = (
    build_and_save_psf_gmm(
        psf8_df,
        "PSF8",
        GMM_GROUPS["PSF8"],
    )
)